### 加载数据

In [1]:
import inspect
import numpy as np
import polars as pl
import pandas as pd
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
from log_result import init_logger, log_result, log_print, read_log
#from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")
init_logger("WalkForward+ParameterSearch+Extremes")   

[log] 已绑定: D:\machine-learning-for-trading\case_studies\lazy_trading\logs\WalkForward+ParameterSearch+Extremes.log


WindowsPath('D:/machine-learning-for-trading/case_studies/lazy_trading/logs/WalkForward+ParameterSearch+Extremes.log')

In [2]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 市场基准
bench_symbol = "510300.SH"
bench = prices[bench_symbol]
prices = prices.drop(columns=[bench_symbol])  # 基准移出资产池
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)
# 中性化
X_net = X.sub(prices_to_returns(bench.to_frame("bench"), drop_inceptions_nan=False)["bench"], axis=0)
# Inf值检查
inf_cols = X_net.columns[np.isinf(X_net).any(axis=0)]
print(inf_cols.tolist())
# 异常收益检查
X_net = X_net.drop(columns=(bad := X_net.columns[(X_net.abs() > 0.25).any()])); 
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

[]
数据异常：收益率超过±30%的列已剔除 3 列 -> ['161811.SZ', '510030.SH', '511580.SH']


### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

中性化后相关性分布整体下移（中位数从高正相关降至 0.143）且正负两端均出现变化：强正相关配对减少约 40%，负相关配对从 0 增至千级。原参数网格（nondomin −0.3~−0.5 空转、correlate 0.1~0.5 高剔除）是按"市场因子主导的强正相关世界"标定的，直接复用确实不适配。建议：nondomin__threshold 收敛至 [-0.5, -0.4]、correlate__threshold 下调至 [0.15, 0.3]，同时计算规模恢复到原始口径水平。

## 相关性分布实测

| 配对相关性分布 | 原始 X | 超额 X_net | 变化 |
|---------------|--------|-----------|------|
| corr > 0.5 | 12,668 对 | 7,791 对 | **−38.5%** |
| corr > 0.3 | 27,712 对 | 16,726 对 | **−39.6%** |
| corr > 0.1 | 42,009 对 | 43,438 对 | +3.4% |
| corr < −0.3 | 0 对 | 1,269 对 | 0 → 1,269 |
| 中位数相关性 | 高（正相关主导） | 0.143 | 整体下移 |

In [3]:
import optuna
from skfolio import Population,MultiPeriodPortfolio
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure,ExtraRiskMeasure
from sklearn.pipeline import Pipeline
from skfolio.metrics import make_scorer
from skfolio.optimization import EqualWeighted
from Pre_selection import DropTailCorrelated
#from skfolio.pre_selection import SelectKExtremes
from Pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated
from skfolio.model_selection import WalkForward
from skfolio.model_selection import cross_val_predict
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
    - purged_size=0：训练结束与测试开始无缝衔接。
    - purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。
    - 建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。

- 训练集扩展与尾部数据处理
    - expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
    - reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。

#### WalkForward + Best Parameter

**① 63/756（k 均为 0.4）**

| 目标函数 | 最优参数 (min_n/thr/corr/k/fitness) | 年化 | 波动 | SR | MDD | Skew | 持仓 |
|---|---|---|---|---|---|---|---|
| ASR−MDD | 20/−0.4/0.4/0.4/mad-avgdd-cvar | 13.97% | 10.49% | 1.33 | 16.53% | −26.71% | 8.6 |
| ASR−2×MDD | 15/−0.4/0.4/0.4/mad-avgdd-cvar | 14.53% | 10.69% | 1.36 | 16.53% | −24.91% | 7.1 |
| **ASR−MDD+skew** (#33) | 15/−0.3/0.4/0.4/mad-avgdd-cvar-sharpe | 13.64% | 10.58% | 1.29 | 16.53% | −24.33% | 7.0 |

**② 63/504（k 均为 0.5）**

| 目标函数 | 最优参数 | 年化 | 波动 | SR | MDD | Skew | 持仓 |
|---|---|---|---|---|---|---|---|
| ASR−MDD | 15/−0.5/0.3/0.5/mad-avgdd-cvar-sharpe | 11.06% | 9.48% | 1.17 | 11.87% | +2.86% | 5.8 |
| **ASR−MDD+skew** (#34) | 15/−0.3/0.2/0.5/mad-avgdd-cvar-sharpe | 11.70% | 10.00% | 1.17 | 13.87% | **+35.53%** | 4.0 |

**③ 63/252（⚠️ k 不同：0.2 vs 0.4，差异混入 k 档效应）**

| 目标函数 | 最优参数 | 年化 | 波动 | SR | MDD | Skew | 持仓 |
|---|---|---|---|---|---|---|---|
| ASR−MDD | 15/−0.4/0.1/0.2/variance-avgdd | 14.94% | 12.85% | 1.16 | 16.78% | −17.43% | 2.6 |
| **ASR−MDD+skew** (#35) | 25/−0.5/0.1/0.4/mad-maxdd-cvar-sharpe | 10.68% | 11.10% | 0.96 | 16.40% | **+3.94%** | 3.2 |

In [21]:
selection_pipe = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("extremes", SelectKExtremes(k=0.5, measure=ExtraRiskMeasure.KURTOSIS, highest=False)),
        ("nondomin", SelectNonDominated(min_n_assets=15, threshold=-0.3,
                        fitness_measures=[PerfMeasure.MEAN,
                        #RiskMeasure.VARIANCE,
                        #RiskMeasure.SEMI_DEVIATION,
                        RiskMeasure.AVERAGE_DRAWDOWN,
                        #RiskMeasure.MEAN_ABSOLUTE_DEVIATION,
                        #RiskMeasure.MAX_DRAWDOWN,
                        RiskMeasure.CVAR,
                        RatioMeasure.SHARPE_RATIO
                        ])),
        ("correlate", DropCorrelated(threshold=0.2, absolute=False)),
    ])

In [22]:
train_portfolios = []
test_portfolios = []
train_portfolios_net = []
test_portfolios_net = []

cv = WalkForward(test_size=252//4, train_size=int(252*2), purged_size=1, reduce_test=True, expand_train=False)
for i, (train_index, test_index) in enumerate(cv.split(X_net)):
    # 划分训练测试集（X 与 X_net 行索引一致）
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    X_net_train = X_net.iloc[train_index]
    X_net_test = X_net.iloc[test_index]

    # 筛选在超额口径 X_net 上执行，列集由 X_net 决定
    X_train = selection_pipe.fit_transform(X_train)
    cols = X_train.columns
    if len(cols) == 0:
        continue

    # 训练在超额口径 X_net 上执行（等权权重与数值无关，仅记录列集）
    m = EqualWeighted(portfolio_params=dict(name="Fold %d" % i)).fit(X_train)

    # X 口径：真实收益
    train_portfolios.append(m.predict(X_train[cols]))
    test_portfolios.append(m.predict(X_test[cols]))
    # X_net 口径：超额收益
    train_portfolios_net.append(m.predict(X_train))
    test_portfolios_net.append(m.predict(X_net_test[cols]))

population_train = Population(train_portfolios)
population_test = Population(MultiPeriodPortfolio(test_portfolios))
population_train_net = Population(train_portfolios_net)
population_test_net = Population(MultiPeriodPortfolio(test_portfolios_net))

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population_train_net.set_portfolio_params(tag="Train (Net)")
population_test_net.set_portfolio_params(tag="Test (Net)")
population = population_train + population_test
population_net = population_train_net + population_test_net

In [23]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [24]:
population_test.plot_cumulative_returns()

In [25]:
MultiPeriodPortfolio(test_portfolios).plot_cumulative_returns()

In [26]:
mpt_summary = MultiPeriodPortfolio(test_portfolios).summary()
mpt_summary

Mean                                     0.035%
Annualized Mean                           8.82%
Variance                                0.0041%
Annualized Variance                       1.03%
Semi-Variance                           0.0020%
Annualized Semi-Variance                  0.51%
Standard Deviation                        0.64%
Annualized Standard Deviation            10.17%
Semi-Deviation                            0.45%
Annualized Semi-Deviation                 7.12%
Mean Absolute Deviation                   0.43%
CVaR at 95%                               1.49%
EVaR at 95%                               2.19%
Worst Realization                         3.90%
CDaR at 95%                              12.46%
MAX Drawdown                             17.48%
Average Drawdown                          3.45%
EDaR at 95%                              13.97%
First Lower Partial Moment                0.21%
Ulcer Index                               0.049
Gini Mean Difference                    

In [ ]:
#log_print(mpt_summary, section="OOS MPP Summary",echo=False)